# Data Cleaning & Transformation

This notebook performs data cleaning and preprocessing for all mutual fund datasets.

Tasks:
- Handle missing values
- Remove duplicates
- Convert date columns
- Standardize column names
- Validate data quality
- Prepare datasets for database loading

## NAV History Data Cleaning

In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/02_nav_history.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["amfi_code", "date"]
)

df = df.drop_duplicates()

df["nav"] = (
    df.groupby("amfi_code")["nav"]
      .ffill()
)

df = df[df["nav"] > 0]

df.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

print("NAV cleaned")

NAV cleaned


## Scheme Performance Cleaning

In [4]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Remove rows with missing values
df = df.dropna()

# Expense Ratio Validation
df = df[
    (df["expense_ratio_pct"] >= 0.1)
    &
    (df["expense_ratio_pct"] <= 2.5)
]

# Morningstar Rating Validation
df = df[
    (df["morningstar_rating"] >= 1)
    &
    (df["morningstar_rating"] <= 5)
]

# Risk Grade Validation
valid_risk = [
    "Low",
    "Moderate",
    "High",
    "Very High"
]

df = df[
    df["risk_grade"].isin(valid_risk)
]

# Flag anomalies
df["anomaly"] = (
    (df["return_1yr_pct"] > 100) |
    (df["return_1yr_pct"] < -50) |
    (df["return_3yr_pct"] > 100) |
    (df["return_3yr_pct"] < -50) |
    (df["return_5yr_pct"] > 150) |
    (df["return_5yr_pct"] < -50)
)

df.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

print("Scheme Performance cleaned successfully")
print("Rows:", len(df))
print("Anomalies:", df["anomaly"].sum())

Scheme Performance cleaned successfully
Rows: 36
Anomalies: 0


## Investor Transaction Cleaning

In [5]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

df["transaction_date"] = pd.to_datetime(
    df["transaction_date"]
)

df["transaction_type"] = (
    df["transaction_type"]
      .str.upper()
)

mapping = {
    "SIP":"SIP",
    "LUMPSUM":"LUMPSUM",
    "REDEMPTION":"REDEMPTION"
}

df["transaction_type"] = (
    df["transaction_type"]
      .map(mapping)
)

df = df[df["amount_inr"] > 0]

valid_kyc = [
    "VERIFIED",
    "PENDING",
    "REJECTED"
]

df = df[
    df["kyc_status"].isin(valid_kyc)
]

df.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

print("Transactions cleaned")

Transactions cleaned


## Cleaning Remaining Datasets

In [6]:
import pandas as pd

# 1. Fund Master

df = pd.read_csv("../data/raw/01_fund_master.csv")

df = df.drop_duplicates()

df["launch_date"] = pd.to_datetime(
    df["launch_date"],
    errors="coerce"
)

df.to_csv(
    "../data/processed/fund_master_clean.csv",
    index=False
)

print("Fund Master cleaned")


# 2. AUM by Fund House

df = pd.read_csv(
    "../data/raw/03_aum_by_fund_house.csv"
)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df = df.drop_duplicates()

df = df[df["aum_crore"] > 0]

df.to_csv(
    "../data/processed/aum_by_fund_house_clean.csv",
    index=False
)

print("AUM cleaned")


# 3. SIP Inflows

df = pd.read_csv(
    "../data/raw/04_monthly_sip_inflows.csv"
)

df = df.drop_duplicates()

df.to_csv(
    "../data/processed/monthly_sip_inflows_clean.csv",
    index=False
)

print("SIP cleaned")


# 4. Category Inflows

df = pd.read_csv(
    "../data/raw/05_category_inflows.csv"
)

df = df.drop_duplicates()

df["category"] = (
    df["category"]
      .str.strip()
      .str.title()
)

df.to_csv(
    "../data/processed/category_inflows_clean.csv",
    index=False
)

print("Category Inflows cleaned")


# 5. Industry Folio Count

df = pd.read_csv(
    "../data/raw/06_industry_folio_count.csv"
)

df = df.drop_duplicates()

df.to_csv(
    "../data/processed/industry_folio_count_clean.csv",
    index=False
)

print("Folio Count cleaned")


# 6. Portfolio Holdings

df = pd.read_csv(
    "../data/raw/09_portfolio_holdings.csv"
)

df = df.drop_duplicates()

df["portfolio_date"] = pd.to_datetime(
    df["portfolio_date"],
    errors="coerce"
)

df = df[
    (df["weight_pct"] >= 0)
    &
    (df["weight_pct"] <= 100)
]

df.to_csv(
    "../data/processed/portfolio_holdings_clean.csv",
    index=False
)

print("Portfolio cleaned")


# 7. Benchmark Indices

df = pd.read_csv(
    "../data/raw/10_benchmark_indices.csv"
)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df = df.drop_duplicates()

df = df[df["close_value"] > 0]

df.to_csv(
    "../data/processed/benchmark_indices_clean.csv",
    index=False
)

print("Benchmark cleaned")




Fund Master cleaned
AUM cleaned
SIP cleaned
Category Inflows cleaned
Folio Count cleaned
Portfolio cleaned
Benchmark cleaned
